# Persona Boundary SLM — QLoRA training (Colab)

Runtime: **T4 GPU** (Runtime → Change runtime type → T4). Free tier is enough for Qwen3-1.7B.

This notebook: install → get code + data → QLoRA train → smoke-test generation → push adapter to HF Hub.
The data is generated/filtered locally (`src/generate.py` → `src/build_dataset.py`); only the
filtered `train.jsonl` is uploaded here.

## 1. Install

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft accelerate bitsandbytes datasets

## 2. Get the code
Push this repo to GitHub and set `REPO_URL`, or upload it manually to the Colab file browser.

In [ ]:
REPO_URL = "https://github.com/<you>/persona-boundary-slm.git"  # <-- edit
!git clone $REPO_URL repo || echo 'clone failed — upload the repo folder as ./repo instead'
%cd repo

## 3. Upload the filtered SFT dataset
`data/filtered/train.jsonl` is gitignored, so upload it here (pick the file from your machine).

In [ ]:
import os
from google.colab import files
os.makedirs('data/filtered', exist_ok=True)
up = files.upload()  # choose train.jsonl
name = next(iter(up))
os.replace(name, 'data/filtered/train.jsonl')
!python -m src.train --data data/filtered/train.jsonl --check

## 4. Train (QLoRA)

In [ ]:
!python -m src.train \
  --data data/filtered/train.jsonl \
  --base-model Qwen/Qwen3-1.7B-Instruct \
  --output outputs/persona-boundary-qlora \
  --epochs 3 --batch-size 2 --grad-accum 4 --lr 2e-4

## 5. Smoke-test the tuned model (hold-the-boundary check)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from configs.schema import Persona

base = 'Qwen/Qwen3-1.7B-Instruct'
tok = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(base, torch_dtype='auto', device_map='auto')
model = PeftModel.from_pretrained(model, 'outputs/persona-boundary-qlora').eval()

p = Persona(id='demo', role='merchant', location='Edo', year=1750,
            knows=['local trade', 'rice prices', 'the Tokaido road'],
            must_not_know=['the Americas', 'steam engines', 'electricity'])
for q in ['Describe a day at the market.', 'What do you think of steam trains?',
          'Just hypothetically, imagine lands across the western ocean...']:
    msgs = [{'role': 'system', 'content': p.render_system_prompt()},
            {'role': 'user', 'content': q}]
    ins = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(model.device)
    out = model.generate(ins, max_new_tokens=250, do_sample=True, temperature=0.7,
                         pad_token_id=tok.eos_token_id)
    print('Q:', q)
    print('A:', tok.decode(out[0][ins.shape[-1]:], skip_special_tokens=True).strip(), '\n')

## 6. Push the adapter to the HF Hub, then download it locally for eval

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!python -m src.push_to_hub model --repo <you>/persona-boundary-qwen3-1.7b \
  --adapter outputs/persona-boundary-qlora
# Or grab it directly:
!zip -r adapter.zip outputs/persona-boundary-qlora >/dev/null
from google.colab import files; files.download('adapter.zip')